# LegacyAgent V1 - Day 3 - Contextual-Biasing Baseline

Idea here: get a free WER improvement with zero training, just by feeding the base Qwen3-ASR model a legal-vocabulary hint through its documented context/hotwords mechanism (a system-role message with `Vocabulary: ...` text, per the Qwen3-ASR-1.7B-hf model card). Then score it with the exact same frozen Day 2 eval script.

Glossary comes from MD's starter legal terms plus case names/speaker surnames pulled from the TRAIN manifest only -- not the eval benchmark, since I don't want to bias the model with vocabulary drawn from the same set it's about to be scored on. I'm deliberately not reusing `entity_terms.json` here either -- its own metadata says it's an evaluation artifact and shouldn't double as this glossary.

In [2]:
# CELL 1
# Mount Drive and locate frozen artifacts
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/legacyagent")
MANIFEST_DIR = PROJECT_DIR / "manifests"
EVAL_DIR = PROJECT_DIR / "eval"
RESULTS_DIR = PROJECT_DIR / "results" / "day3_biasing"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

benchmark_path = EVAL_DIR / "frozen_test_benchmark.jsonl"
train_manifest_path = MANIFEST_DIR / "train_manifest.jsonl"
entity_terms_path = EVAL_DIR / "entity_terms.json"

print("Project dir      :", PROJECT_DIR.exists())
print("Frozen benchmark :", benchmark_path.exists())
print("Train manifest   :", train_manifest_path.exists())
print("Entity terms     :", entity_terms_path.exists())

Mounted at /content/drive
Project dir      : True
Frozen benchmark : True
Train manifest   : True
Entity terms     : True


In [3]:
# CELL 2
# Load the frozen Day 2 evaluation benchmark and normalizer
# (verbatim from Day 2 Cell 4 -- this evaluator is frozen for all V1 comparisons)
import json
import re
import unicodedata

!pip -q install jiwer
from jiwer import wer


def normalize_for_wer(text: str) -> str:
    """
    Frozen LegacyAgent WER normalization.
    Do not modify after baseline results are produced.
    """
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()
    text = text.replace("’", "'").replace("‘", "'")
    text = re.sub(r"[‐-‒–—−]+", " ", text)
    text = re.sub(r"[^\w\s']", " ", text)
    text = text.replace("'", "")
    text = re.sub(r"\s+", " ", text).strip()
    return text


benchmark_rows = []
with open(benchmark_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            benchmark_rows.append(json.loads(line))

print("Frozen benchmark segments:", len(benchmark_rows))
print("Frozen held-out cases    :", len({row['case_id'] for row in benchmark_rows}))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.2 MB/s eta 0:00:00
Frozen benchmark segments: 1163
Frozen held-out cases    : 5


In [4]:
# CELL 3
# Build the Day 3 bias glossary from the TRAIN manifest ONLY.
#
# entity_terms.json is deliberately not reused here -- it is built from the
# frozen eval benchmark itself and its own metadata states it "must not
# automatically become the Day 3 bias glossary." Sourcing this list from the
# train split instead keeps the bias-prompt vocabulary independent of the
# eval set it will be scored against.
import json
re_module = re  # already imported above

starter_legal_terms = [
    "certiorari",
    "amicus curiae",
    "stare decisis",
    "sua sponte",
]

train_rows = []
with open(train_manifest_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            train_rows.append(json.loads(line))

print("Train manifest segments:", len(train_rows))

case_names = sorted({
    row["case_name"].strip()
    for row in train_rows
    if row.get("case_name", "").strip()
})

speaker_surnames = set()
for row in train_rows:
    speaker_name = row.get("speaker_name", "").strip()
    if not speaker_name or speaker_name.lower() == "unknown":
        continue
    cleaned_name = re.sub(r",?\s+(Jr\.?|Sr\.?|II|III|IV)$", "", speaker_name, flags=re.IGNORECASE)
    parts = cleaned_name.split()
    if parts:
        surname = parts[-1].strip("., ")
        if len(surname) >= 2:
            speaker_surnames.add(surname)

speaker_surnames = sorted(speaker_surnames)

glossary_terms = []
seen_normalized = set()

for term in starter_legal_terms + case_names + speaker_surnames:
    norm = normalize_for_wer(term)
    if not norm or norm in seen_normalized:
        continue
    seen_normalized.add(norm)
    glossary_terms.append(term)

BIAS_VOCABULARY_TEXT = "Vocabulary: " + ", ".join(glossary_terms) + "."

print("Starter legal terms:", len(starter_legal_terms))
print("Case names (train)  :", len(case_names))
print("Speaker surnames (train):", len(speaker_surnames))
print("Total unique glossary terms:", len(glossary_terms))
print("\nBias prompt text (first 500 chars):")
print(BIAS_VOCABULARY_TEXT[:500])

glossary_record = {
    "project": "LegacyAgent V1",
    "purpose": "Day 3 contextual-biasing prompt vocabulary",
    "sources": ["MD starter legal terms", "train_manifest.jsonl case_name", "train_manifest.jsonl speaker_name"],
    "derived_from_eval_benchmark": False,
    "terms": glossary_terms,
}
with open(RESULTS_DIR / "day3_bias_glossary.json", "w", encoding="utf-8") as f:
    json.dump(glossary_record, f, indent=2, ensure_ascii=False)
print("\nSaved glossary:", RESULTS_DIR / "day3_bias_glossary.json")

Train manifest segments: 3702
Starter legal terms: 4
Case names (train)  : 12
Speaker surnames (train): 46
Total unique glossary terms: 62

Bias prompt text (first 500 chars):
Vocabulary: certiorari, amicus curiae, stare decisis, sua sponte, Ashcroft v. Al-Kidd, Behrens v. Pelletier, Devenpeck v. Alford, Filarsky v. Delia, Hope v. Pelzer, Johnson v. Jones, Lane v. Franks, Messerschmidt v. Millender, Morse v. Frederick, Plumhoff v. Rickard, Reichle v. Howards, Ziglar v. Abbasi, Alito, Bash, Breyer, Coates, Comey, Forrester, Gallagher, Gelernt, Gershengorn, Ginsburg, Hart, Jones, Kagan, Katyal, Kennedy, Kneedler, Lamken, Lane, Mcgill, Meeropol, Mertz, Millett, Mosley, O

Saved glossary: /content/drive/MyDrive/legacyagent/results/day3_biasing/day3_bias_glossary.json


In [5]:
# CELL 4
# Load base Qwen3-ASR for inference (verbatim from Day 2 Cell 8 -- no fine-tune)
import torch
from transformers import AutoProcessor, Qwen3ASRForConditionalGeneration

MODEL_ID = "Qwen/Qwen3-ASR-1.7B-hf"

print("Loading processor...")
qwen_processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

print("Loading base Qwen3-ASR...")
qwen_model = Qwen3ASRForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
qwen_model.eval()

print("\nModel loaded")
print("Model ID :", MODEL_ID)
print("Training :", qwen_model.training)
print("Device   :", next(qwen_model.parameters()).device)

Loading processor...


processor_config.json:   0%|          | 0.00/487 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.08GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]


Model loaded
Model ID : Qwen/Qwen3-ASR-1.7B-hf
Training : False
Device   : cuda:0


In [6]:
# CELL 5
# Output parser (verbatim from Day 2 Cell 10)
import re

def parse_qwen_asr_output(raw_output: str) -> str:
    text = raw_output.strip()
    if "<asr_text>" in text:
        text = text.split("<asr_text>", 1)[1]
    text = re.sub(r"<\|.*?\|>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

print("Parser loaded")

Parser loaded


In [7]:
# CELL 6
# Biasing inference function.
#
# Identical to the Day 2 base-model transcribe function EXCEPT a system-role
# message carrying the bias vocabulary is prepended before the user audio
# turn, per the officially documented Qwen3-ASR-1.7B-hf "Context / hotwords"
# mechanism (huggingface.co/Qwen/Qwen3-ASR-1.7B-hf). Generation call, parser,
# and dtype handling are unchanged from Day 2 to keep the eval methodology
# identical apart from this one input-construction difference.
import torch

def transcribe_qwen_segment_biased(row, waveform, sample_rate, bias_text):
    start_sample = int(row["start"] * sample_rate)
    end_sample = int(row["end"] * sample_rate)
    audio_segment = waveform[:, start_sample:end_sample]
    audio_array = audio_segment.squeeze(0).numpy()

    conversation = [
        {"role": "system", "content": [{"type": "text", "text": bias_text}]},
        {"role": "user", "content": [{"type": "audio", "audio": audio_array}]},
    ]

    inputs = qwen_processor.apply_chat_template(
        conversation, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt"
    )

    fixed_inputs = {}
    for key, value in inputs.items():
        if isinstance(value, torch.Tensor):
            value = value.to(qwen_model.device)
            if value.is_floating_point():
                value = value.to(torch.bfloat16)
            fixed_inputs[key] = value
        else:
            fixed_inputs[key] = value

    with torch.inference_mode():
        generated_ids = qwen_model.generate(**fixed_inputs, max_new_tokens=256, do_sample=False)

    prompt_length = fixed_inputs["input_ids"].shape[1]
    generated_only = generated_ids[:, prompt_length:]
    raw_output = qwen_processor.batch_decode(generated_only, skip_special_tokens=True)[0].strip()
    transcript = parse_qwen_asr_output(raw_output)

    return raw_output, transcript

print("Biasing inference function loaded")

Biasing inference function loaded


In [8]:
import torchaudio

test_row = benchmark_rows[0]
waveform, sr = torchaudio.load(test_row["audio_path"])
raw, transcript = transcribe_qwen_segment_biased(test_row, waveform, sr, BIAS_VOCABULARY_TEXT)
print("REF :", test_row["text"])
print("PRED:", transcript)

REF : We'll hear argument first this morning in Case No. 15-118, Hernandez v. Mesa. Mr. Hilliard.
PRED: You'll hear argument first this morning in case fifteen one eighteen Hernandez versus Mesa, Mr. Hilliard.


In [9]:
# CELL 7
# Checkpointed full biasing inference over the frozen eval benchmark
# (mirrors Day 2 Cell 11/13 pattern)
import json, time
import torchaudio

BIASING_RESULTS_PATH = RESULTS_DIR / "qwen3_asr_biasing_predictions.jsonl"

def load_existing_predictions(path):
    if not path.exists():
        return [], set()
    rows = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
    completed = {r["segment_id"] for r in rows if r.get("status") == "success"}
    return rows, completed

def save_prediction(path, result):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")

existing_rows, completed_ids = load_existing_predictions(BIASING_RESULTS_PATH)
remaining_rows = [r for r in benchmark_rows if r["segment_id"] not in completed_ids]

print("Total segments:", len(benchmark_rows))
print("Already completed:", len(completed_ids))
print("Remaining:", len(remaining_rows))

cached_case_id = None
cached_waveform = None
cached_sample_rate = None
success_count = 0
failure_count = 0

for i, row in enumerate(remaining_rows, start=1):
    if row["case_id"] != cached_case_id:
        cached_waveform, cached_sample_rate = torchaudio.load(row["audio_path"])
        assert cached_sample_rate == 16000
        cached_case_id = row["case_id"]
        print(f"\nLoaded case: {cached_case_id}")

    t0 = time.time()
    try:
        raw_output, transcript = transcribe_qwen_segment_biased(
            row, cached_waveform, cached_sample_rate, BIAS_VOCABULARY_TEXT
        )
        result = {
            "segment_id": row["segment_id"], "case_id": row["case_id"],
            "case_name": row["case_name"], "start": row["start"], "end": row["end"],
            "duration": row["duration"], "reference": row["text"], "prediction": transcript,
            "raw_output": raw_output, "inference_seconds": round(time.time() - t0, 3),
            "model": MODEL_ID, "status": "success",
        }
        success_count += 1
    except Exception as e:
        result = {
            "segment_id": row["segment_id"], "case_id": row["case_id"],
            "reference": row["text"], "error": str(e), "model": MODEL_ID, "status": "error",
        }
        failure_count += 1

    save_prediction(BIASING_RESULTS_PATH, result)
    if i % 10 == 0:
        print(f"Processed {i}/{len(remaining_rows)}  (success={success_count}, failed={failure_count})")

print("\nDone. Success:", success_count, "Failed:", failure_count)

Total segments: 1163
Already completed: 0
Remaining: 1163

Loaded case: 2016_15-118
Processed 10/1163  (success=10, failed=0)
Processed 20/1163  (success=20, failed=0)
Processed 30/1163  (success=30, failed=0)
Processed 40/1163  (success=40, failed=0)
Processed 50/1163  (success=50, failed=0)
Processed 60/1163  (success=60, failed=0)
Processed 70/1163  (success=70, failed=0)
Processed 80/1163  (success=80, failed=0)
Processed 90/1163  (success=90, failed=0)
Processed 100/1163  (success=100, failed=0)
Processed 110/1163  (success=110, failed=0)
Processed 120/1163  (success=120, failed=0)
Processed 130/1163  (success=130, failed=0)
Processed 140/1163  (success=140, failed=0)
Processed 150/1163  (success=150, failed=0)
Processed 160/1163  (success=160, failed=0)
Processed 170/1163  (success=170, failed=0)
Processed 180/1163  (success=180, failed=0)
Processed 190/1163  (success=190, failed=0)
Processed 200/1163  (success=200, failed=0)
Processed 210/1163  (success=210, failed=0)

Loaded ca

In [10]:
# CELL 8
# Official corpus WER for the biasing run (identical computation shape to Day 2/6)
from jiwer import wer
import json

results = [json.loads(l) for l in open(BIASING_RESULTS_PATH, encoding="utf-8") if l.strip()]
biasing_by_id = {r["segment_id"]: r for r in results if r.get("status") == "success"}

references, hypotheses = [], []
for row in benchmark_rows:
    seg_id = row["segment_id"]
    if seg_id not in biasing_by_id:
        continue
    references.append(normalize_for_wer(row["text"]))
    hypotheses.append(normalize_for_wer(biasing_by_id[seg_id]["prediction"]))

biasing_wer = wer(references, hypotheses)
print("QWEN3-ASR + CONTEXTUAL BIASING -- OFFICIAL RESULT")
print("=" * 60)
print("Evaluation segments:", len(references))
print("CORPUS WER (%)     :", round(biasing_wer * 100, 2))

QWEN3-ASR + CONTEXTUAL BIASING -- OFFICIAL RESULT
Evaluation segments: 1163
CORPUS WER (%)     : 7.57


In [11]:
# CELL 9
# Entity-span WER for the biasing run, using the frozen Day 2 entity_terms.json
# (that file IS meant for eval-scoring -- only the BIAS PROMPT in Cell 3 avoided it)
import json

with open(entity_terms_path) as f:
    entity_terms_data = json.load(f)

candidate_terms = [t["term"].lower() for t in entity_terms_data["terms"]]
print("Entity vocabulary size:", len(candidate_terms))

def extract_entity_spans(text, terms):
    text_norm = normalize_for_wer(text)
    found = []
    for term in terms:
        term_norm = normalize_for_wer(term)
        if term_norm and term_norm in text_norm:
            found.append(term_norm)
    return found

entity_refs, entity_hyps = [], []
for row in benchmark_rows:
    seg_id = row["segment_id"]
    if seg_id not in biasing_by_id:
        continue
    reference_text = row["text"]
    prediction_text = biasing_by_id[seg_id]["prediction"]

    ref_spans = extract_entity_spans(reference_text, candidate_terms)
    hyp_spans = extract_entity_spans(prediction_text, candidate_terms)
    if ref_spans:
        entity_refs.append(" ".join(ref_spans))
        entity_hyps.append(" ".join(hyp_spans) if hyp_spans else "")

entity_wer_biasing = wer(entity_refs, entity_hyps) if entity_refs else None
print("Biasing entity-span WER:", round(entity_wer_biasing * 100, 2) if entity_wer_biasing is not None else "n/a", "%")
print("Segments with entity mentions:", len(entity_refs))

Entity vocabulary size: 24
Biasing entity-span WER: 27.27 %
Segments with entity mentions: 163


In [12]:
# CELL 10
# Save Day 3 results and compare against the frozen Day 2 base-model numbers
import json
from datetime import datetime, timezone

# Fill these in from your saved Day 2 results file / notebook output
baseline_overall_wer_pct = 7.71   # Qwen3-ASR base, from Day 2
baseline_entity_wer_pct = 30.67   # Qwen3-ASR base, from Day 2

comparison = {
    "model": "Qwen3-ASR-1.7B + contextual biasing (zero training)",
    "glossary_term_count": len(glossary_terms),
    "glossary_source": "MD starter terms + train_manifest case_name/speaker_name",
    "overall_wer_pct": round(biasing_wer * 100, 2),
    "entity_span_wer_pct": round(entity_wer_biasing * 100, 2) if entity_wer_biasing is not None else None,
    "baseline_overall_wer_pct": baseline_overall_wer_pct,
    "baseline_entity_wer_pct": baseline_entity_wer_pct,
    "overall_wer_relative_improvement_pct": round(
        (baseline_overall_wer_pct - biasing_wer * 100) / baseline_overall_wer_pct * 100, 2
    ),
    "entity_wer_relative_improvement_pct": round(
        (baseline_entity_wer_pct - entity_wer_biasing * 100) / baseline_entity_wer_pct * 100, 2
    ) if entity_wer_biasing is not None else None,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
}

print(json.dumps(comparison, indent=2))

with open(RESULTS_DIR / "day3_comparison.json", "w") as f:
    json.dump(comparison, f, indent=2)

print("\nSaved:", RESULTS_DIR / "day3_comparison.json")

{
  "model": "Qwen3-ASR-1.7B + contextual biasing (zero training)",
  "glossary_term_count": 62,
  "glossary_source": "MD starter terms + train_manifest case_name/speaker_name",
  "overall_wer_pct": 7.57,
  "entity_span_wer_pct": 27.27,
  "baseline_overall_wer_pct": 7.71,
  "baseline_entity_wer_pct": 30.67,
  "overall_wer_relative_improvement_pct": 1.77,
  "entity_wer_relative_improvement_pct": 11.08,
  "completed_at_utc": "2026-08-31T18:46:35.794776+00:00"
}

Saved: /content/drive/MyDrive/legacyagent/results/day3_biasing/day3_comparison.json
